In [ ]:
#model logging +.json logging + versioning

import json
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col
import pandas as pd
import pickle
import os
import base64
from datetime import datetime
import tempfile
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry

# ---------------------------
# Snowflake session
# ---------------------------
session = get_active_session()

# ---------------------------
# Custom Model Wrapper
# ---------------------------
class InterceptCustomModel(custom_model.CustomModel):
    def __init__(self, model):
        super().__init__(context=None)
        self.model = model

    @custom_model.inference_api
    def predict(self, input: pd.DataFrame) -> pd.DataFrame:
        output = self.model.predict(
            data=input, 
            coeffs=self.model.model_coefficients_final
        )
        return pd.DataFrame({'prediction': output})

# ---------------------------
# Initialize Registry
# ---------------------------
reg = Registry(
    session=session,
    database_name="ORANGE_ZONE_SBX_TA",
    schema_name="REGISTRY"
)

git_commit = os.getenv("GIT_COMMIT")
git_branch = os.getenv("GIT_BRANCH")
git_repo = os.getenv("GIT_REPO")

if not git_commit:
    git_commit = "UNKNOWN"

# ---------------------------
# Fetch all intercept rows
# ---------------------------
intercept_rows = session.table("PUBLIC.PROD_FINAL_MODEL_LASSO_COEFFICIENTS") \
    .filter(col("PARAMETERS") == "Intercept") \
    .select(
        "REGIONNAME", "F_CODE", "MODEL_BYTES",
        "RSQ", "WMAPE", "MAPE", "RMSE", "BIAS", "TRACKING_SIGNAL",
        "ALPHA", "LAMBDA",
        "TRAIN_START_DATE", "TRAIN_END_DATE",
        "LOAD_TS"
    ).collect()

# ---------------------------
# Loop over all models
# ---------------------------
for row in intercept_rows:
    region = row["REGIONNAME"]
    f_code = row["F_CODE"]
    region_fcode = f"{region}_{f_code}"
    print(f"Processing model: {region_fcode}")

    # Decode model
    model_bytes = base64.b64decode(row["MODEL_BYTES"])
    model_object = pickle.loads(model_bytes)

    # Instantiate custom model
    custom_model_instance = InterceptCustomModel(model_object)

    # Get all features for this region_fcode
    features_rows = session.table("PUBLIC.PROD_FINAL_MODEL_LASSO_COEFFICIENTS") \
        .filter((col("REGIONNAME") == region) & (col("F_CODE") == f_code)) \
        .select("PARAMETERS") \
        .collect()
    features = [f["PARAMETERS"] for f in features_rows]

    # ---------------------------
    # Determine version
    # ---------------------------
    # Check if previous metadata exists
    try:
        with open(f"model_metadata_{region_fcode}.json", "r") as f:
            last_metadata = json.load(f)
        last_features = last_metadata["features"]
        last_train_period = last_metadata["train_period"]
        last_version = last_metadata.get("version", "v_0_0")
        major, minor = map(int, last_version.replace("v_", "").split("_"))

        if features != last_features:
            major += 1
            minor = 0  # reset minor
        elif (str(row["TRAIN_START_DATE"]) != last_train_period["start"]) or \
             (str(row["TRAIN_END_DATE"]) != last_train_period["end"]):
            minor += 1
        # else keep same version
        version = f"v_{major}_{minor}"

    except FileNotFoundError:
        # First version
        version = "v_0_0"
        
    region_fcode_safe = region_fcode.replace("-", "_")

    # ---------------------------
    # Create model metadata dict
    # ---------------------------
    model_metadata = {
        "name": region_fcode,
        "model_type": "ElasticNet",
        "description": f"ElasticNet demand model for {region_fcode}",
        "version": version,
        "source": {
            "git_commit": git_commit,
            "git_branch": git_branch,
            "git_repo": git_repo
        },
        "metrics": {
            "rsq": float(row["RSQ"]),
            "wmape": float(row["WMAPE"]),
            "mape": float(row["MAPE"]),
            "rmse": float(row["RMSE"]),
            "bias": float(row["BIAS"]),
            "tracking_signal": float(row["TRACKING_SIGNAL"]),
        },
        "hyperparameters": {
            "alpha": float(row["ALPHA"]),
            "lambda": float(row["LAMBDA"]),
        },
        "train_period": {
            "start": str(row["TRAIN_START_DATE"]),
            "end": str(row["TRAIN_END_DATE"]),
        },
        "features": features,
        "training_date": str(row["LOAD_TS"]),
        "tags": {
            "region": region,
            "f_code": f_code,
            "algorithm": "ElasticNet"
        }
    }

    # ---------------------------
    # Save model metadata to JSON file
    # ---------------------------
    

    tmp_dir = tempfile.gettempdir()
    artifact_path = os.path.join(
        tmp_dir, f"model_metadata_{region_fcode_safe}.json"
    )
    
    with open(artifact_path, "w") as f:
        json.dump(model_metadata, f, indent=2)

    model_name=f"intercept_model_{region_fcode_safe}"
    
    reg.delete_model(model_name=model_name)
    mv = reg.log_model(
    model=custom_model_instance,
    model_name=f"intercept_model_{region_fcode_safe}",
    # version_name=version,
    # version_name=registry_version,
    conda_dependencies=["scikit-learn", "pandas"],
    options={"relax_version": False},
    user_files={"metadata": [artifact_path]}, 
    comment=f"Custom Intercept Model for {region_fcode}",
    sample_input_data=pd.DataFrame(model_object.X)
)
    # assign alias to new version
    # mv.set_alias(f"semantic_{version}")

    print(f"Successfully logged model and metadata for {region_fcode} with version {version}")
